# Pre- vs. post-SHINE cohort comparison

Statistically compares the pre-SHINE and post-SHINE SpAM cohorts -- data quality, response
dynamics, and the resulting behavioral RDMs -- for **retained subjects only** (production
cohort, `status == "full data"`, `task_version >= 4.0`; the same selection
`analysis.rdms.behavioral.build_behavioral_rdm()` uses, see `_select_participants()`).
Data-prep and figure helpers live in `analysis/behavioral/cohort_comparison.py`; this notebook
loads the data, runs the statistical tests, and renders the figures -- matching this repo's
existing notebook-as-analysis-surface convention (`analysis/pilot/analysis.ipynb`,
`analysis/behavioral/noise_ceiling.ipynb`).

All pairwise comparisons are two-sided Mann-Whitney U tests on per-subject aggregates (one
value per subject, avoiding pseudo-replication across trials), with rank-biserial effect size
*r* = 1 - 2U/(n_A x n_B); |r| >= 0.5 is conventionally large. Categorical comparisons (sex,
screening outcome) use Fisher's exact test. Colours are the validated colorblind-safe
blue/orange pair already used for v1.0/v2.0 in `analysis/pilot/figures.py`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/behavioral/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, spearmanr, ks_2samp, wasserstein_distance

from analysis.utils.load_data import load_data
from analysis.pilot.simulate_null_distances import simulate as simulate_null_distances
from analysis.rdms.behavioral import build_behavioral_rdm
from analysis.behavioral.cohort_comparison import (
    retained_participants, pair_coverage, subject_reliability, main_trials,
    subject_trial_stats, pairwise_distance_values, subject_wasserstein_distance,
    subject_ks_distance, mwu_two_group,
    classical_mds_eigenspectrum, effective_rank, mds_sweep, masked_kendall_spearman,
    procrustes_m2, MDS_CACHE_DIR,
    fig_reliability, fig_trial_time_moves, fig_trial_progression, fig_distance_vs_null,
    fig_demographics, fig_retention_status,
    fig_eigenvalue_scree, fig_stress_scree, fig_cross_cohort_agreement,
)

_COHORTS = ("pre", "post")
_LABEL = {"pre": "Pre-SHINE", "post": "Post-SHINE"}


def _print_mwu(result: dict, alpha: float = 0.05) -> None:
    sig = "significant" if result["p"] < alpha else "not significant"
    print(f"  {result['comparison']}: U={result['U']:.1f}  p={result['p']:.4f} ({sig})  r={result['r']:.3f}")


def _print_fisher(label: str, table: list, oddsratio: float, p: float, alpha: float = 0.05) -> None:
    sig = "significant" if p < alpha else "not significant"
    print(f"  {label}: table={table}  odds ratio={oddsratio:.3f}  p={p:.4f} ({sig})")

## 0. Load data

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
data = load_data(cache_dir)
df_participants, df_trials = data["participants"], data["trials"]

participants = {c: retained_participants(df_participants, c) for c in _COHORTS}
participant_ids = {c: participants[c]["participant_id"] for c in _COHORTS}
trials_main = {c: main_trials(df_trials, participant_ids[c]) for c in _COHORTS}

for c in _COHORTS:
    print(f"{_LABEL[c]}: {len(participants[c])} retained subjects, {len(trials_main[c])} main trials")

Pre-SHINE: 79 retained subjects, 1738 main trials
Post-SHINE: 76 retained subjects, 1672 main trials


## 1. General statistics

Number of retained subjects, coverage of the full 725-image pair space by at least one SpAM
judgement, mean +/- SEM number of ratings per (covered) image-pair, and within-subject
reliability (per-subject mean Spearman r across that subject's verbatim trial repeats).

In [3]:
coverage = {c: pair_coverage(df_trials, participant_ids[c], c) for c in _COHORTS}
reliability = {c: subject_reliability(df_trials, participant_ids[c]) for c in _COHORTS}

general_stats = pd.DataFrame({
    _LABEL[c]: {
        "N subjects": len(participants[c]),
        "% image-pairs rated >=1x": round(coverage[c]["pct_covered"], 2),
        "Ratings/pair, mean": round(coverage[c]["mean_per_pair"], 3),
        "Ratings/pair, SEM": round(coverage[c]["sem_per_pair"], 4),
        "Reliability, median": round(float(reliability[c].median()), 3),
        "Reliability, mean": round(float(reliability[c].mean()), 3),
        "Reliability, SEM": round(float(reliability[c].std(ddof=1) / np.sqrt(len(reliability[c]))), 4),
    }
    for c in _COHORTS
}).T
general_stats

,N subjects,% image-pairs rated >=1x,"Ratings/pair, mean","Ratings/pair, SEM","Reliability, median","Reliability, mean","Reliability, SEM"
Pre-SHINE,79.0,64.57,1.949,0.0027,0.276,0.319,0.0168
Post-SHINE,76.0,63.09,1.919,0.0027,0.291,0.312,0.0139


## 2. Within-subject reliability

Distribution of within-subject reliability (per-subject mean Spearman r across trial repeats)
for each cohort, and a Mann-Whitney U test of whether the two cohorts differ.

In [4]:
fig_reliability(reliability).show()

In [5]:
result_reliability = mwu_two_group(reliability["pre"], reliability["post"], _LABEL["pre"], _LABEL["post"])
_print_mwu(result_reliability)
pd.DataFrame([result_reliability])

  Pre-SHINE vs Post-SHINE: U=2898.0  p=0.7110 (not significant)  r=0.035


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
0,Pre-SHINE vs Post-SHINE,79,76,0.2762,0.2912,0.3192,0.312,0.1495,0.121,2898.0,0.711035,0.0346


## 3. Pairwise distance distributions vs. the random-assignment null

Per-cohort distribution of pairwise SpAM distances (main trials, all retained subjects) against
the null distribution of pairwise distances under uniformly-random placement (same simulation as
`analysis/pilot/simulate_null_distances.py`, `num_dots=20` matching `images_per_trial`). Distance
from null is quantified two ways: **Wasserstein (earth mover's) distance is the primary measure**
-- it integrates over the full gap between the trial's distance distribution and the null, so it's
sensitive to *how far apart* the two distributions are, not just whether their shapes differ. **KS
distance (D statistic) is kept as a secondary/shape-robustness check** (it only captures the
maximum CDF gap). Both are compared between cohorts with a Mann-Whitney U test on per-subject means.

We also check, descriptively, whether each cohort's *pooled* distances actually differ from the
null at all (not just from each other) via a single Wasserstein distance + KS test per cohort
against the null sample. This pools all of a cohort's trial-level pairwise-distance observations
into one sample, so the resulting KS p-value is not a valid independent-samples test (observations
within a trial/subject aren't independent) -- it's reported as a directional/magnitude check, not
a formal inference.

In [6]:
null_distribution = simulate_null_distances(num_dots=20, num_trials=1000, seed=42)
dists = {c: pairwise_distance_values(trials_main[c]) for c in _COHORTS}
wasserstein_dist = {c: subject_wasserstein_distance(trials_main[c], null_distribution) for c in _COHORTS}
ks_distance = {c: subject_ks_distance(trials_main[c], null_distribution) for c in _COHORTS}

fig_distance_vs_null(dists, null_distribution, wasserstein_dist, ks_distance).show()

In [7]:
for c in _COHORTS:
    w = wasserstein_distance(dists[c], null_distribution)
    D, p = ks_2samp(dists[c], null_distribution)
    print(f"  {_LABEL[c]} vs null (pooled, descriptive): Wasserstein={w:.4f}  KS D={D:.4f}  KS p={p:.2e}")

  Pre-SHINE vs null (pooled, descriptive): Wasserstein=0.0274  KS D=0.0553  KS p=4.75e-321


  Post-SHINE vs null (pooled, descriptive): Wasserstein=0.0277  KS D=0.0589  KS p=0.00e+00


In [8]:
result_wasserstein = mwu_two_group(wasserstein_dist["pre"], wasserstein_dist["post"], _LABEL["pre"], _LABEL["post"])
result_ks = mwu_two_group(ks_distance["pre"], ks_distance["post"], _LABEL["pre"], _LABEL["post"])
print("Primary (Wasserstein):")
_print_mwu(result_wasserstein)
print("Secondary (KS):")
_print_mwu(result_ks)
pd.DataFrame([result_wasserstein, result_ks], index=["Wasserstein (primary)", "KS (secondary)"])

Primary (Wasserstein):
  Pre-SHINE vs Post-SHINE: U=2784.0  p=0.4363 (not significant)  r=0.073
Secondary (KS):
  Pre-SHINE vs Post-SHINE: U=2805.0  p=0.4818 (not significant)  r=0.066


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
Wasserstein (primary),Pre-SHINE vs Post-SHINE,79,76,0.0400,0.0412,0.0454,0.0476,0.0218,0.0231,2784.0,0.436265,0.0726
KS (secondary),Pre-SHINE vs Post-SHINE,79,76,0.1307,0.1327,0.1421,0.1463,0.0548,0.0557,2805.0,0.481839,0.0656


## 4. Trial duration and number of moves

Per-trial distributions of trial duration and number of moves (main trials only -- catch
trials use a different `images_per_trial`, see `SpAM_Task/task_config.json`, so mixing them in
would confound the move-count comparison), the per-subject moves-vs-duration relationship, and
how both measures trend over the course of the session (mean +/- SE per trial number, with thin
per-subject lines -- same progression-panel style as `analysis/pilot/figures.py`'s
`fig_trial_duration`/`fig_moves`). Mann-Whitney U tests below use per-subject means.

In [9]:
subj_trial_stats = {c: subject_trial_stats(trials_main[c]) for c in _COHORTS}
fig_trial_time_moves(trials_main, subj_trial_stats).show()

In [10]:
fig_trial_progression(trials_main).show()

In [11]:
result_rt = mwu_two_group(subj_trial_stats["pre"]["rt_s"], subj_trial_stats["post"]["rt_s"], _LABEL["pre"], _LABEL["post"])
result_moves = mwu_two_group(subj_trial_stats["pre"]["num_moves"], subj_trial_stats["post"]["num_moves"], _LABEL["pre"], _LABEL["post"])
_print_mwu(result_rt)
_print_mwu(result_moves)

for c in _COHORTS:
    rho, p = spearmanr(subj_trial_stats[c]["num_moves"], subj_trial_stats[c]["rt_s"])
    print(f"  {_LABEL[c]} moves-vs-duration (per subject): Spearman rho={rho:.3f}  p={p:.4f}")

pd.DataFrame([result_rt, result_moves], index=["Trial duration (s)", "Number of moves"])

  Pre-SHINE vs Post-SHINE: U=3089.0  p=0.7569 (not significant)  r=-0.029
  Pre-SHINE vs Post-SHINE: U=2682.0  p=0.2528 (not significant)  r=0.107
  Pre-SHINE moves-vs-duration (per subject): Spearman rho=0.305  p=0.0062
  Post-SHINE moves-vs-duration (per subject): Spearman rho=0.138  p=0.2334


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
Trial duration (s),Pre-SHINE vs Post-SHINE,79,76,79.1208,78.9177,86.8618,84.2233,22.4874,19.2530,3089.0,0.756853,-0.0290
Number of moves,Pre-SHINE vs Post-SHINE,79,76,24.6364,25.7955,27.0984,27.2967,9.7595,6.4409,2682.0,0.252777,0.1066


## 5. Age and gender distributions

Age (Mann-Whitney U) and sex (Fisher's exact test on the Male/Female counts) distributions for
each cohort's retained subjects.

In [12]:
fig_demographics(participants).show()

In [13]:
result_age = mwu_two_group(participants["pre"]["age"], participants["post"]["age"], _LABEL["pre"], _LABEL["post"])
_print_mwu(result_age)

sex_counts = {c: participants[c]["sex"].value_counts() for c in _COHORTS}
sex_table = [
    [int(sex_counts["pre"].get("Male", 0)), int(sex_counts["post"].get("Male", 0))],
    [int(sex_counts["pre"].get("Female", 0)), int(sex_counts["post"].get("Female", 0))],
]
odds_ratio, p_sex = fisher_exact(sex_table)
_print_fisher("Sex (Male/Female x cohort)", sex_table, odds_ratio, p_sex)

pd.DataFrame([result_age], index=["Age"])

  Pre-SHINE vs Post-SHINE: U=2899.5  p=0.7148 (not significant)  r=0.034
  Sex (Male/Female x cohort): table=[[42, 38], [37, 37]]  odds ratio=1.105  p=0.8719 (not significant)


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
Age,Pre-SHINE vs Post-SHINE,79,76,29.0,29.0,30.6835,31.7237,8.9364,11.0883,2899.5,0.714755,0.0341


## 6. Additional comparisons

Two comparisons that speak directly to whether the retained pre- and post-SHINE samples are
equivalent, beyond what was asked for above:

- **Screening/retention outcome**: among *all* `task_version >= 4.0` production participants
  (not just retained ones), does the rate of ending up "full data" vs. screened out / flagged
  as a false positive on the full-session recheck differ by cohort? A differential retention
  rate would mean the two retained samples aren't drawn from equivalently-screened pools.
- **QC flag rate**: per-subject rate of the recorded main-trial `qc_flag` (see
  `SpAM_Task/js/utils.js`'s `computeMainQcFlag`), compared between cohorts -- a proxy for
  whether one SHINE variant's images were harder to arrange sensibly.

In [14]:
v4plus = df_participants[pd.to_numeric(df_participants["task_version"], errors="coerce") >= 4.0]
v4plus = v4plus[v4plus["cohort"] == "production"]

fig_retention_status(v4plus).show()

In [15]:
full_data = v4plus["status"] == "full data"
retention_table = [
    [int((full_data & (v4plus["shine_variant"] == "pre")).sum()), int((full_data & (v4plus["shine_variant"] == "post")).sum())],
    [int((~full_data & (v4plus["shine_variant"] == "pre")).sum()), int((~full_data & (v4plus["shine_variant"] == "post")).sum())],
]
odds_ratio, p_retention = fisher_exact(retention_table)
_print_fisher("Retained (full data) vs not, by cohort", retention_table, odds_ratio, p_retention)

  Retained (full data) vs not, by cohort: table=[[79, 76], [22, 18]]  odds ratio=0.850  p=0.7238 (not significant)


In [16]:
subj_qc_rate = {
    c: trials_main[c].groupby("participant_id")["qc_flag"].mean()
    for c in _COHORTS
}
result_qc = mwu_two_group(subj_qc_rate["pre"], subj_qc_rate["post"], _LABEL["pre"], _LABEL["post"])
_print_mwu(result_qc)
pd.DataFrame([result_qc])

  Pre-SHINE vs Post-SHINE: U=3412.0  p=0.0136 (significant)  r=-0.137


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
0,Pre-SHINE vs Post-SHINE,79,76,0.0,0.0,0.0109,0.0036,0.0233,0.0144,3412.0,0.013564,-0.1366


## 7. MDS-based structural comparison

So far every comparison has been descriptive (data quality, response dynamics, raw distance
magnitudes). This section instead asks whether the pre- and post-SHINE cohorts induce the same
*perceptual similarity structure* -- i.e. whether SHINE preprocessing changes not just how
subjects behave, but what similarity geometry their judgements imply.

Each cohort's full 725x725 behavioral RDM is built via `analysis.rdms.behavioral.build_behavioral_rdm()`
(`aggregation="simple"`, the project's validated default per `noise_ceiling.ipynb`) -- a condensed
distance vector with NaN for the ~35-37% of pairs no retained subject in that cohort ever judged.

In [17]:
rdm = {c: build_behavioral_rdm(f"{c}_shine", cache_dir=cache_dir, aggregation="simple") for c in _COHORTS}

[behavioral] variant='pre_shine': 79 participants.


[behavioral] Built 1738 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 64.57% covered (92977 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.95, sd=1.13.
[behavioral] WARNING: 92977 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_pre_simple.npy  (length 262450)


[behavioral] variant='post_shine': 76 participants.


[behavioral] Built 1672 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 63.09% covered (96882 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.92, sd=1.11.
[behavioral] WARNING: 96882 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_post_simple.npy  (length 262450)


### 7.1 Eigenvalue scree (classical MDS, raw RDM)

**Caveat -- read before interpreting this plot.** Classical (Torgerson) MDS eigenvalues require a
*complete* distance matrix, so the ~36% of missing pairs are mean-imputed first. This repo's own
`SpAM_Simulations/empirical/gt_construction.py` measured that this is invalid at this exact
coverage level: mean-imputed pairs are mutually equidistant, which manufactures rank rather than
revealing it (a synthetic **rank-8** ground truth put through the identical mask+fill reported an
**effective rank of ~193**, statistically indistinguishable from noise). Our own numbers below land
in the same regime. This plot is kept as a literal, qualitative view per the original ask -- it is
**not** a validated dimensionality estimate. Section 7.3's SMACOF stress scree (no imputation) is
the project's validated alternative.

In [18]:
MAX_DIMS = 15
MDS_DIMS = list(range(2, MAX_DIMS + 1))

eigenvalues = {c: classical_mds_eigenspectrum(rdm[c]) for c in _COHORTS}
for c in _COHORTS:
    print(f"  {_LABEL[c]}: effective rank (mean-imputed classical MDS) = {effective_rank(eigenvalues[c])} of 725 -- see caveat above")

fig_eigenvalue_scree(eigenvalues, MAX_DIMS).show()

  Pre-SHINE: effective rank (mean-imputed classical MDS) = 354 of 725 -- see caveat above
  Post-SHINE: effective rank (mean-imputed classical MDS) = 353 of 725 -- see caveat above


### 7.2 Weighted-SMACOF embeddings (R `smacof` backend)

Metric SMACOF via R's `smacof::mds()` (`SpAM_Simulations/core/multi_dimensional_scaling.py`'s
`run_mds()`), with binary observed/unobserved weights -- unlike classical MDS, missing pairs are
simply excluded from the fit, no imputation. One fit per dimensionality 2..`MAX_DIMS` per cohort
(28 fits total); each real-scale fit takes ~15-30s, so results are cached to
`analysis/results/behavioral/mds_cache/` (one `.npz` per cohort x dimension) and re-runs of this
notebook reuse the cache instead of refitting.

In [19]:
mds_results = {
    c: mds_sweep(rdm[c], MDS_DIMS, c, cache_dir=MDS_CACHE_DIR, max_iters=500, convergence_tol=1e-6)
    for c in _COHORTS
}
for c in _COHORTS:
    print(f"  {_LABEL[c]}: {len(mds_results[c])} dimensionalities ready (cached under {MDS_CACHE_DIR})")

  Pre-SHINE: 14 dimensionalities ready (cached under C:\Users\nirjo\Documents\University\PhD\Projects\hierarchical_image_database\.claude\worktrees\awesome-leakey-d0b3a3\analysis\results\behavioral\mds_cache)
  Post-SHINE: 14 dimensionalities ready (cached under C:\Users\nirjo\Documents\University\PhD\Projects\hierarchical_image_database\.claude\worktrees\awesome-leakey-d0b3a3\analysis\results\behavioral\mds_cache)


### 7.3 Stress scree plot

Weighted-SMACOF stress-1 across dimensionality, one trace per cohort -- the validated
(no-imputation) dimensionality diagnostic, in contrast to section 7.1's eigenvalue scree.

In [20]:
fig_stress_scree(mds_results, MDS_DIMS).show()

### 7.4 Pre- vs. post-SHINE structural agreement

For the raw RDMs and for each embedding dimensionality, do the pre- and post-SHINE cohorts agree
on the *same* similarity structure? Kendall tau and Spearman rho are computed on the raw RDMs too
(restricted to jointly-covered pairs), but Procrustes m^2 needs a complete point configuration, so
it's only defined once an MDS embedding exists (`NaN` for "raw RDM" below). Procrustes m^2 is
shown as `1 - m^2` so all three traces read "bigger = better".

In [21]:
agreement: dict[str, dict] = {}
agreement["raw RDM"] = {**masked_kendall_spearman(rdm["pre"], rdm["post"]), "procrustes_m2": None}
for d in MDS_DIMS:
    ks = masked_kendall_spearman(mds_results["pre"][d]["confdist"], mds_results["post"][d]["confdist"])
    m2 = procrustes_m2(mds_results["pre"][d]["conf"], mds_results["post"][d]["conf"])
    agreement[str(d)] = {**ks, "procrustes_m2": m2}

agreement_table = pd.DataFrame(agreement).T[["kendall", "spearman", "procrustes_m2", "n_pairs"]]
agreement_table

,kendall,spearman,procrustes_m2,n_pairs
raw RDM,0.147864,0.219826,NaN,107040.0
2,0.502793,0.689974,0.305794,262450.0
3,0.435604,0.613894,0.375806,262450.0
4,0.404671,0.573814,0.442914,262450.0
5,0.406591,0.574581,0.477917,262450.0
6,0.402729,0.567673,0.550900,262450.0
7,0.418474,0.586165,0.509305,262450.0
8,0.429431,0.599686,0.527540,262450.0
9,0.436127,0.606954,0.534334,262450.0
10,0.447335,0.620495,0.535048,262450.0


In [22]:
fig_cross_cohort_agreement(agreement, MDS_DIMS).show()